# Notebook 23 — Application 1 Stage 2: development information set

Stage 2 constructs leakage-safe development hidden-event samples and a common 10:00-to-10:00 variance horizon. It defines neither a trading rule nor strategy PnL. TEST economic predictors remain sealed.

In [1]:
from pathlib import Path
from datetime import time
from zoneinfo import ZoneInfo
import ast, math, random
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd()/"CODEWORK"/"Linus' Task Re-Do"] if (p/"Data"/"processed").exists())
DATA, PROCESSED, INTERIM = ROOT/"Data", ROOT/"Data"/"processed", ROOT/"Data"/"interim"
NY_TZ, NY = "America/New_York", ZoneInfo("America/New_York")
TRAIN_A_SHARE, TRAIN_B_SHARE, TRAIN_C_SHARE = .50, .25, .25
MIN_MODEL_SUPPORT, ACTIVE_THRESHOLD = 3, .95
REGULAR_HORIZON_MAX_CALENDAR_DAYS, MIN_BRIDGE_CLASS_N = 1, 30
TEST_STAGE2_FEATURES_UNLOCKED = False
EMBARGO_MESSAGE = "TEST Stage 2 economic predictors are sealed until the Stage 3 policy is frozen."
torch.set_num_threads(1)
assert TEST_STAGE2_FEATURES_UNLOCKED is False

def guard(sample_split):
    if str(sample_split).lower()=="test" and not TEST_STAGE2_FEATURES_UNLOCKED:
        raise RuntimeError(EMBARGO_MESSAGE)

def local_time(day, clock):
    return pd.Timestamp(day).tz_localize(NY_TZ)+pd.Timedelta(hours=clock.hour, minutes=clock.minute)

def first_after(dates, day):
    return next((d for d in dates if d > day), None)

## 1. Frozen inputs, Stage 1 contract, and A/B/C eligibility audit

The audit is deliberately completed before any development detector fit or candidate classification.

In [2]:
meta=pd.read_csv(PROCESSED/"22_stage1_run_metadata.csv").set_index("field")["value"]
def parse_clock(value):
    return time.fromisoformat(str(value).split()[0])
ASSUMED_IV_TIME=parse_clock(meta["assumed_iv_time"]); ASSUMED_EXPIRY_TIME=parse_clock(meta["assumed_expiry_time"])
assert meta["test_economics_unlocked"].lower()=="false"
panel=pd.read_csv(PROCESSED/"16_empirical_analysis_panel.csv",parse_dates=["model_day"]).sort_values("model_day").reset_index(drop=True)
panel["canonical_index"]=np.arange(len(panel), dtype=int)
flags=pd.read_csv(PROCESSED/"18_event_day_flags.csv",parse_dates=["model_day"])
mapping=pd.read_csv(PROCESSED/"18_target_specific_event_mapping.csv",parse_dates=["realised_model_day","implied_model_day","target_model_day","event_timestamp_utc","event_timestamp_ny"])
selected_specs=pd.read_csv(PROCESSED/"19_selected_hyperparameters.csv")
stage1_test=pd.read_csv(PROCESSED/"22_stage1_test_feasibility_mapping.csv",parse_dates=["signal_model_day","signal_time_ny","option_entry_time_ny","option_entry_time_utc","expiry_time_ny","expiry_time_utc"])
n21eps=pd.read_csv(PROCESSED/"21_hidden_event_episodes.csv",parse_dates=["start_day","end_day"])
assert len(panel.loc[panel.sample_split.eq("train")])==3626
panel["Q"]=pd.to_numeric(panel["squared_return"]); panel["R"]=pd.to_numeric(panel["realised_variance_ann_252"]); panel["I"]=pd.to_numeric(panel["iv_model_var"])
FEATURES=[f"{state}_lag{lag}" for lag in range(12,0,-1) for state in "QRI"]
for state in "QRI":
    for lag in range(1,13): panel[f"{state}_lag{lag}"]=panel[state].shift(lag)
panel=panel.merge(flags,on="model_day",how="left",validate="one_to_one")
for target in "RI":
    panel[f"{target}_is_target_event_day"]=panel[f"{target}_is_target_event_day"].fillna(False).astype(bool)
    panel[f"{target}_base36"]=np.isfinite(panel[[target,*FEATURES]]).all(axis=1)
train=panel.loc[panel.sample_split.eq("train")].copy().reset_index(drop=True)
nA=math.floor(TRAIN_A_SHARE*len(train)); nB=math.floor(TRAIN_B_SHARE*len(train))
train["abc_block"]=np.select([train.index<nA,train.index<nA+nB],["A","B"],default="C")
panel=panel.merge(train[["model_day","abc_block"]],on="model_day",how="left")
assert train.abc_block.value_counts().to_dict()=={"A":1813,"B":906,"C":907}
assert set(train.abc_block)=={"A","B","C"} and train.index.is_unique
known_R=set(mapping.loc[mapping.target.eq("R"),"target_model_day"]); known_I=set(mapping.loc[mapping.target.eq("I"),"target_model_day"])
panel["known_pair"]=panel.model_day.isin(known_R)|panel.model_day.shift(1).isin(known_I)
train=panel.loc[panel.sample_split.eq("train")].copy().reset_index(drop=True)
audit=[]
for b in "ABC":
    x=train.loc[train.abc_block.eq(b)]
    row={"block":b,"start_day":x.model_day.min(),"end_day":x.model_day.max(),"canonical_model_day_rows":len(x),
         "Q_nonmissing":int(x.Q.notna().sum()),"R_nonmissing":int(x.R.notna().sum()),"I_nonmissing":int(x.I.notna().sum()),
         "known_primary_target_rows":int(x.known_pair.sum())}
    for t in "RI":
        base=x[f"{t}_base36"]; ordinary=base&~x[f"{t}_is_target_event_day"]&~x.known_pair
        row[f"{t}_background_eligible"]=int((base&~x[f"{t}_is_target_event_day"]).sum())
        row[f"{t}_ordinary_ratio_eligible"]=int(ordinary.sum())
        row[f"{t}_model_specific_ratio_count"]=int(ordinary.sum())*4
    row["prospective_RI_observed_pair_count"]=int((x.R.notna()&x.I.notna()&~x.known_pair).sum())
    audit.append(row)
abc_audit=pd.DataFrame(audit)
abc_audit.to_csv(PROCESSED/"23_stage2_abc_eligibility_audit.csv",index=False)
display(abc_audit)

,block,start_day,end_day,canonical_model_day_rows,Q_nonmissing,R_nonmissing,I_nonmissing,known_primary_target_rows,R_background_eligible,R_ordinary_ratio_eligible,R_model_specific_ratio_count,I_background_eligible,I_ordinary_ratio_eligible,I_model_specific_ratio_count,prospective_RI_observed_pair_count
0,A,2003-05-05,2010-04-14,1813,1724,1724,1489,428,708,681,2724,707,527,2108,1082
1,B,2010-04-15,2013-10-03,906,755,755,906,241,407,390,1560,418,304,1216,556
2,C,2013-10-04,2017-03-27,907,893,893,907,228,568,548,2192,581,440,1760,667


## 2. Inherited model specifications and expanding-window OOS detector

Architectures/hyperparameters and neural fixed epoch counts are inherited from N19. Each stage refits parameters/scalers on historical eventless targets only; no forecast block is used for early stopping or epoch selection.

In [3]:
specs=selected_specs.loc[selected_specs.model_family.isin(["RANDOM_FOREST","GRADIENT_BOOSTING","MLP","TRANSFORMER"])].copy()
specs["inheritance_statement"]="Frozen N19 specifications; stage-specific parameters/scalers refit on historical eventless targets only; no hyperparameter reselection."
specs["neural_training_convention"]="Fixed N19 final epochs; no B/C/VALIDATION early stopping."
specs.to_csv(PROCESSED/"23_stage2_inherited_model_specifications.csv",index=False)
RF_FIXED=dict(n_estimators=500,max_features=.5,criterion="squared_error",bootstrap=True,random_state=19,n_jobs=-1)
GB_FIXED=dict(max_depth=2,min_samples_leaf=10,subsample=1.,loss="squared_error",random_state=19)
SEEDS=[19,119,219]; BATCH=64
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
class MLP(nn.Module):
    def __init__(self,h):
        super().__init__(); layers=[]; w=36
        for z in h: layers += [nn.Linear(w,z),nn.ReLU(),nn.Dropout(.1)]; w=z
        layers += [nn.Linear(w,1)]; self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(-1)
class TR(nn.Module):
    def __init__(self,d,h,f):
        super().__init__(); self.emb=nn.Linear(3,d); self.pos=nn.Parameter(torch.zeros(1,12,d),requires_grad=False)
        pos=torch.arange(12).unsqueeze(1); div=torch.exp(torch.arange(0,d,2)*(-math.log(10000.)/d))
        self.pos[:,:,0::2]=torch.sin(pos*div); self.pos[:,:,1::2]=torch.cos(pos*div)
        self.mask=torch.triu(torch.ones(12,12,dtype=torch.bool),1)
        self.enc=nn.TransformerEncoder(nn.TransformerEncoderLayer(d,h,f,dropout=.1,activation="gelu",batch_first=True),1); self.head=nn.Linear(d,1)
    def forward(self,x): return self.head(self.enc(self.emb(x)+self.pos,mask=self.mask)[:,-1]).squeeze(-1)
def scale_fit(x,t):
    d={}; 
    for state in "QRI":
        z=x[[f"{state}_lag{k}" for k in range(1,13)]].to_numpy(float).ravel(); d[state]=(z.mean(),z.std())
    y=x[t].to_numpy(float); d["Y"]=(y.mean(),y.std()); return d
def Xscale(x,sc):
    a=x[FEATURES].to_numpy(float).copy()
    for j,f in enumerate(FEATURES): a[:,j]=(a[:,j]-sc[f[0]][0])/sc[f[0]][1]
    return a
def neural_fit(fam,param,epochs,x,t,sc,seed):
    set_seed(seed); m=MLP(param[0]) if fam=="MLP" else TR(*param[0]); opt=torch.optim.AdamW(m.parameters(),lr=param[1],weight_decay=1e-4)
    xx=torch.tensor(Xscale(x,sc),dtype=torch.float32); 
    if fam=="TRANSFORMER": xx=xx.reshape(-1,12,3)
    yy=torch.tensor((x[t].to_numpy(float)-sc["Y"][0])/sc["Y"][1],dtype=torch.float32); loader=DataLoader(TensorDataset(xx,yy),batch_size=BATCH,shuffle=True)
    for _ in range(int(epochs)):
        m.train()
        for a,b in loader:
            opt.zero_grad(); loss=nn.MSELoss()(m(a),b); loss.backward()
            if fam=="TRANSFORMER": torch.nn.utils.clip_grad_norm_(m.parameters(),1.)
            opt.step()
    return m
def neural_pred(models,x,sc,fam):
    a=torch.tensor(Xscale(x,sc),dtype=torch.float32)
    if fam=="TRANSFORMER": a=a.reshape(-1,12,3)
    with torch.no_grad(): y=np.mean([m.eval()(a).numpy() for m in models],axis=0)
    return y*sc["Y"][1]+sc["Y"][0]
def cfg(t,f):
    r=specs.loc[(specs.target.eq(t))&(specs.model_family.eq(f))].iloc[0]
    return ast.literal_eval(r.selected_hyperparameters), r.final_fixed_epochs
def stage_forecasts(history_blocks, forecast_block, label):
    history=train.loc[train.abc_block.isin(history_blocks)]; future=panel.loc[(panel.abc_block.eq(forecast_block)) if forecast_block in "ABC" else panel.sample_split.eq("validation")].copy()
    rows=[]
    for t in "RI":
        tr=history.loc[history[f"{t}_base36"]&~history[f"{t}_is_target_event_day"]].copy()
        te=future.loc[future[f"{t}_base36"]].copy()
        assert tr.model_day.max()<te.model_day.min()
        for fam in ["RANDOM_FOREST","GRADIENT_BOOSTING"]:
            params,_=cfg(t,fam); model=(RandomForestRegressor(**RF_FIXED,**params) if fam=="RANDOM_FOREST" else GradientBoostingRegressor(**GB_FIXED,**params)); model.fit(tr[FEATURES],tr[t]); pred=model.predict(te[FEATURES])
            rows += [{"model_day":d,"target":t,"model":fam,"actual":a,"forecast":p,"is_target_event_day":e} for d,a,p,e in zip(te.model_day,te[t],pred,te[f"{t}_is_target_event_day"])]
        sc=scale_fit(tr,t)
        for fam in ["MLP","TRANSFORMER"]:
            params,epochs=cfg(t,fam); models=[neural_fit(fam,params,epochs,tr,t,sc,s) for s in SEEDS]; pred=neural_pred(models,te,sc,fam)
            rows += [{"model_day":d,"target":t,"model":fam,"actual":a,"forecast":p,"is_target_event_day":e} for d,a,p,e in zip(te.model_day,te[t],pred,te[f"{t}_is_target_event_day"])]
    return pd.DataFrame(rows).assign(fit_history="+".join(history_blocks),forecast_block=label)
forecast_B=stage_forecasts(["A"],"B","B"); forecast_C=stage_forecasts(["A","B"],"C","C"); forecast_V=stage_forecasts(["A","B","C"],"validation","VALIDATION")
assert set(forecast_B.model)==set(forecast_C.model)==set(forecast_V.model)=={"RANDOM_FOREST","GRADIENT_BOOSTING","MLP","TRANSFORMER"}

In [4]:
def pct(v,dist):
    n=len(dist); q=np.searchsorted(dist,v,side="right"); return np.clip((q+.5)/(n+1),.5/(n+1),(n+.5)/(n+1))
def block_chronology(block):
    x=train.loc[train.abc_block.eq(block)] if block in "ABC" else panel.loc[panel.sample_split.eq("validation")]
    return x[["model_day","canonical_index"]].drop_duplicates().sort_values("canonical_index").copy()
def model_cdf_distributions(forecasts):
    ordinary=forecasts.loc[~forecasts.is_target_event_day&forecasts.forecast.gt(0)&forecasts.actual.notna()].copy()
    ordinary["ratio"]=ordinary.actual/ordinary.forecast
    return {(t,m):np.sort(g.ratio.to_numpy(float)) for (t,m),g in ordinary.groupby(["target","model"])}
def target_consensus(forecasts,dist):
    rows=[]
    for (d,t),z in forecasts.loc[forecasts.forecast.gt(0)&forecasts.actual.notna()].groupby(["model_day","target"]):
        vals=[pct(a/p,dist[(t,m)]) for a,p,m in zip(z.actual,z.forecast,z.model)]
        if len(vals)>=MIN_MODEL_SUPPORT: rows.append({"model_day":d,"target":t,"P":np.median(vals),"n_models":len(vals)})
    return pd.DataFrame(rows)
def canonical_pairs(forecasts,dist,block):
    scores=target_consensus(forecasts,dist)
    r=scores.loc[scores.target.eq("R"),["model_day","P","n_models"]].rename(columns={"P":"P_R","n_models":"n_models_R"})
    i=scores.loc[scores.target.eq("I"),["model_day","P","n_models"]].rename(columns={"P":"P_I","n_models":"n_models_I"})
    canonical=block_chronology(block).merge(r,on="model_day",how="left",validate="one_to_one").merge(i,on="model_day",how="left",validate="one_to_one")
    canonical["P_I_prev"]=canonical.P_I.shift(1); canonical["implied_reference_model_day"]=canonical.model_day.shift(1); canonical["implied_reference_canonical_index"]=canonical.canonical_index.shift(1)
    pairs=canonical.loc[canonical.P_R.notna()&canonical.P_I_prev.notna()].copy()
    assert (pairs.canonical_index-pairs.implied_reference_canonical_index).eq(1).all()
    expected=canonical.set_index("canonical_index").model_day
    assert all(expected.loc[int(r.implied_reference_canonical_index)]==r.implied_reference_model_day for r in pairs.itertuples())
    pairs["Z_R"]=norm.ppf(pairs.P_R); pairs["Z_I_prev"]=norm.ppf(pairs.P_I_prev); pairs["D"]=pairs.Z_R-pairs.Z_I_prev
    pairs["is_known_primary_event_pair"]=pairs.model_day.isin(known_R)|pairs.implied_reference_model_day.isin(known_I)
    return scores,canonical,pairs
def calibration(forecasts,block,next_block):
    dist=model_cdf_distributions(forecasts); scores,canonical,pairs=canonical_pairs(forecasts,dist,block)
    ordinary=pairs.loc[~pairs.is_known_primary_event_pair].copy()
    assert len(ordinary)>=250, f"Detector support insufficient: {block}={len(ordinary)}"
    lo,hi=ordinary.D.quantile([.01,.99])
    summary=[]
    for (t,m),a in dist.items(): summary.append({"calibration_block":block,"target":t,"model":m,"ordinary_ratio_n":len(a),"consensus_pair_n":len(ordinary),"q_low":lo,"q_high":hi,"fit_history_used_for_calibration_forecasts":{"B":"A","C":"A+B"}[block],"next_detection_block":next_block})
    return scores,canonical,ordinary,dist,lo,hi,pd.DataFrame(summary)
B_scores,B_canonical,B_pairs,B_dist,qloB,qhiB,calB=calibration(forecast_B,"B","C")
def score_detect(forecasts,dist,lo,hi,role):
    scores,canonical,pairs=canonical_pairs(forecasts,dist,"C" if role=="C" else "VALIDATION")
    pairs["candidate_type"]=np.select([pairs.D.ge(hi)&pairs.P_R.ge(ACTIVE_THRESHOLD),pairs.D.le(lo)&pairs.P_I_prev.ge(ACTIVE_THRESHOLD)],["REALISED_ONLY","IMPLIED_ONLY"],default="NONE")
    pairs.loc[pairs.is_known_primary_event_pair,"candidate_type"]="NONE"; pairs["period"]=role
    return pairs
C_days=score_detect(forecast_C,B_dist,qloB,qhiB,"C")
C_scores,C_canonical,C_pairs,C_dist,qloC,qhiC,calC=calibration(forecast_C,"C","VALIDATION")
V_days=score_detect(forecast_V,C_dist,qloC,qhiC,"VALIDATION")
for block,scores,pairs in [("B",B_scores,B_pairs),("C",C_scores,C_pairs)]:
    for target in "RI":
        abc_audit.loc[abc_audit.block.eq(block),f"actual_{target}_flexible_score_count"]=int(scores.loc[scores.target.eq(target)].shape[0])
    abc_audit.loc[abc_audit.block.eq(block),"actual_ge3_model_consensus_pair_count"]=len(pairs)
    abc_audit.loc[abc_audit.block.eq(block),"actual_canonical_detector_calibration_pair_count"]=len(pairs)
abc_audit.to_csv(PROCESSED/"23_stage2_abc_eligibility_audit.csv",index=False)
def collapse(x,prefix):
    q=x.loc[x.candidate_type.ne("NONE")].sort_values("canonical_index").copy(); q["new"]=q.candidate_type.ne(q.candidate_type.shift())|q.canonical_index.diff().ne(1); q["num"]=q["new"].cumsum(); q["episode_id"]=prefix+q["num"].astype(str).str.zfill(3)
    rows=[]
    for eid,g in q.groupby("episode_id"):
        peak=g.loc[g.D.abs().idxmax()]; rows.append({**peak.to_dict(),"episode_id":eid,"start_day":g.model_day.min(),"end_day":g.model_day.max(),"n_days":len(g)})
    return q,pd.DataFrame(rows)
C_candidates,C_eps=collapse(C_days,"E23_C_"); V_candidates,V_eps=collapse(V_days,"E23_V_")
candidate_days=pd.concat([C_candidates,V_candidates],ignore_index=True); episodes=pd.concat([C_eps,V_eps],ignore_index=True)
cal=pd.concat([calB,calC],ignore_index=True); cal.to_csv(PROCESSED/"23_stage2_detector_calibration_summary.csv",index=False)
candidate_days.to_csv(PROCESSED/"23_stage2_candidate_days.csv",index=False); episodes.to_csv(PROCESSED/"23_stage2_episodes.csv",index=False)


## 3. Stage 1 timing reconstruction, ordinary bridge, and development information set

TEST rows are reconstructed only as safe timestamps and availability fields. All functions producing economic predictors are guarded.

In [5]:
iv=pd.read_csv(ROOT/"archive_notebook14_results"/"usdjpy_atm_tidy.csv",parse_dates=["date"])
iv=iv.loc[(iv.currency_pair=="USDJPY")&(iv.tenor=="1D")&(iv.ticker=="USDJPYVON BGN Curncy")&pd.to_numeric(iv.atm_iv,errors="coerce").gt(0)].copy(); iv["date_key"]=iv.date.dt.date
h=pd.read_csv(INTERIM/"usdjpy_hourly_mid_audited.csv",parse_dates=["timestamp_parsed"]); h["timestamp_parsed"]=pd.to_datetime(h.timestamp_parsed,utc=True); h["end"]=h.timestamp_parsed+pd.Timedelta(hours=1); h["end_ny"]=h.end.dt.tz_convert(NY_TZ); h["date_key"]=h.end_ny.dt.date
b=pd.read_csv(DATA/"usdjpy_hourly_data"/"usdjpy_hourly_bid.csv"); a=pd.read_csv(DATA/"usdjpy_hourly_data"/"usdjpy_hourly_ask.csv"); b["timestamp_parsed"]=pd.to_datetime(b.timestamp,utc=True); a["timestamp_parsed"]=pd.to_datetime(a.timestamp,utc=True)
h=h.merge(b[["timestamp_parsed","close"]].rename(columns={"close":"bid"}),on="timestamp_parsed").merge(a[["timestamp_parsed","close"]].rename(columns={"close":"ask"}),on="timestamp_parsed"); h["valid_quote"]=h.ask.ge(h.bid)&h.bid.gt(0)&h.ask.gt(0)&h.mid_close.gt(0)
def quotes(clock):
    return h.loc[(h.end_ny.dt.hour.eq(clock.hour))&(h.end_ny.dt.minute.eq(clock.minute))&(h.end_ny.dt.second.eq(clock.second))&h.valid_quote&h.end_ny.dt.dayofweek.lt(5)].copy()
entryq,expiryq=quotes(ASSUMED_IV_TIME),quotes(ASSUMED_EXPIRY_TIME); eligible=sorted(set(entryq.date_key)&set(iv.date_key)); expiry_dates=sorted(set(expiryq.date_key))
def timing(signal_day, split):
    e=first_after(eligible,signal_day); return e,first_after(expiry_dates,e) if e else None
SAFE_TEST_COLUMNS=["episode_id","candidate_type","signal_model_day","signal_time","entry_date","entry_time","expiry_date","expiry_time","evaluable","unevaluable_reason"]
def reconstruct_safe_test_contract():
    rows=[]
    for ep in stage1_test.itertuples():
        signal=pd.Timestamp(ep.signal_model_day).date(); e,x=timing(signal,"test")
        rows.append({"episode_id":ep.episode_id,"candidate_type":ep.candidate_type,"signal_model_day":pd.Timestamp(signal),"signal_time":local_time(signal,time(17)),"entry_date":e,"entry_time":local_time(e,ASSUMED_IV_TIME) if e else pd.NaT,"expiry_date":x,"expiry_time":local_time(x,ASSUMED_EXPIRY_TIME) if x else pd.NaT,"evaluable":bool(e and x),"unevaluable_reason":np.nan if e and x else "No later date has both same-date raw O/N ATM IV and exact observed 10:00 NY spot close."})
    return pd.DataFrame(rows)
reconstructed_test_safe=reconstruct_safe_test_contract()
frozen_test_safe=stage1_test.rename(columns={"signal_time_ny":"signal_time","option_entry_time_ny":"entry_time","expiry_time_ny":"expiry_time"})[SAFE_TEST_COLUMNS].copy()
for col in SAFE_TEST_COLUMNS:
    left=reconstructed_test_safe[col].astype(object).where(reconstructed_test_safe[col].notna(),"<MISSING>").astype(str); right=frozen_test_safe[col].astype(object).where(frozen_test_safe[col].notna(),"<MISSING>").astype(str); assert left.equals(right),f"Stage 1 safe timestamp contract mismatch: {col}"
event_cols=["event_id","family","event_timestamp_original","event_timestamp_utc","event_timestamp_ny","realised_model_day"]
events=mapping.drop_duplicates("event_id")[event_cols].copy()
events["source_timestamp_ny"]=pd.to_datetime(events.event_timestamp_original,utc=True,errors="coerce",format="mixed").dt.tz_convert(NY_TZ)
events["event_date_ny"]=events.source_timestamp_ny.dt.date
events.loc[events.event_date_ny.isna(),"event_date_ny"]=pd.to_datetime(events.loc[events.event_date_ny.isna(),"realised_model_day"]).dt.date
events["timestamp_status"]=np.where(events.source_timestamp_ny.notna(),"SOURCE_TIMESTAMP_AVAILABLE","DATE_ONLY_OR_TIMING_UNRESOLVED")
def horizon_events(t0,t1):
    exact=events.loc[events.source_timestamp_ny.notna()&(events.source_timestamp_ny>t0)&(events.source_timestamp_ny<=t1)]
    unresolved=events.loc[events.source_timestamp_ny.isna()&events.event_date_ny.between(t0.date(),t1.date())]
    all_events=pd.concat([exact,unresolved]).drop_duplicates("event_id").sort_values(["event_date_ny","event_id"])
    return {"known_event_in_horizon":bool(len(all_events)),"n_known_events":len(all_events),"known_event_families":" | ".join(all_events.family.astype(str)),"known_event_timestamps":" | ".join(all_events.source_timestamp_ny.astype(str).where(all_events.source_timestamp_ny.notna(),"DATE_ONLY")),"known_event_timestamp_statuses":" | ".join(all_events.timestamp_status),"event_timing_ambiguous":bool(len(unresolved)),"exact_known_event_in_horizon":bool(len(exact))}
def path_var(e,x,sample_split):
    guard(sample_split); t0=local_time(e,ASSUMED_IV_TIME).tz_convert("UTC"); t1=local_time(x,ASSUMED_EXPIRY_TIME).tz_convert("UTC"); p=h.loc[h.end.between(t0,t1)].sort_values("end")
    if len(p)<2 or not p.valid_quote.all(): return np.nan,"invalid_quote"
    if p.iloc[1:].gap_classification.isin(["short unexpected weekday/other gap","major historical discontinuity","other interval"]).any(): return np.nan,"unexpected_gap"
    return float(np.square(np.diff(np.log(p.mid_close.to_numpy(float)))).sum()),""
def sparse_forecasts(hist,future):
    tr=train.loc[train.abc_block.isin(hist)&train.R.notna()&train.R.shift(1).notna()&train.I.shift(1).notna()&~train.R_is_target_event_day].copy()
    fit=sm.OLS(tr.R,sm.add_constant(tr[["R_lag1","I_lag1"]])).fit()
    return future[["model_day"]].assign(R_background_forecast=fit.predict(sm.add_constant(future[["R_lag1","I_lag1"]],has_constant="add")).to_numpy())
sfB=sparse_forecasts(["A"],train.loc[train.abc_block.eq("B")]); sfC=sparse_forecasts(["A","B"],train.loc[train.abc_block.eq("C")]); sfV=sparse_forecasts(["A","B","C"],panel.loc[panel.sample_split.eq("validation")]); sf=pd.concat([sfB.assign(block="B"),sfC.assign(block="C"),sfV.assign(block="VALIDATION")])
def build_stage2_economic_row(ep,role,sample_split):
    guard(sample_split)
    d=pd.Timestamp(ep.start_day).date(); e,x=timing(d,sample_split); block_dates=set(train.loc[train.abc_block.eq("C"),"model_day"].dt.date) if role=="POLICY_TRAIN" else set(panel.loc[panel.sample_split.eq("validation"),"model_day"].dt.date)
    base={"development_role":role,"episode_id":ep.episode_id,"candidate_type":ep.candidate_type,"signal_model_day":d,"signal_time":local_time(d,time(17)),"entry_date":e,"expiry_date":x,"economically_evaluable":False,"unevaluable_reason":""}
    if not e or not x or e not in block_dates or x not in block_dates: base["unevaluable_reason"]="Entry or expiry falls outside the same development block."; return base
    f=sf.loc[sf.model_day.dt.date.eq(e),"R_background_forecast"]
    actual,reason=path_var(e,x,sample_split)
    if len(f)!=1 or reason or not np.isfinite(f.iloc[0]) or f.iloc[0]<=0: base["unevaluable_reason"]=reason or "Nonpositive/unavailable OOS realised forecast."; return base
    t0,t1=local_time(e,ASSUMED_IV_TIME),local_time(x,ASSUMED_EXPIRY_TIME); ann=horizon_events(t0,t1)
    base.update({"entry_time":t0,"expiry_time":t1,"elapsed_calendar_hours":(t1-t0).total_seconds()/3600,"T":(t1-t0).total_seconds()/(365*86400),"horizon_class":"REGULAR" if (x-e).days<=REGULAR_HORIZON_MAX_CALENDAR_DAYS else "EXTENDED","R_background_forecast":f.iloc[0],"R_forecast_total_variance_unbridged":f.iloc[0]/252,"actual_option_horizon_realised_variance":actual,**{k:v for k,v in ann.items() if k!="exact_known_event_in_horizon"},"economically_evaluable":True})
    return base
all_eps=pd.concat([C_eps.assign(development_role="POLICY_TRAIN"),V_eps.assign(development_role="POLICY_VALIDATION")],ignore_index=True)
raw=pd.DataFrame([build_stage2_economic_row(x,x.development_role,"development") for x in all_eps.itertuples()]).drop(columns=["Index"],errors="ignore")
bridge_rows=[]
for block,days,fore in [("B",train.loc[train.abc_block.eq("B"),"model_day"],sfB),("C",train.loc[train.abc_block.eq("C"),"model_day"],sfC),("VALIDATION",panel.loc[panel.sample_split.eq("validation"),"model_day"],sfV)]:
    block_dates=set(days.dt.date)
    for e in sorted(set(eligible)&block_dates):
        x=first_after(expiry_dates,e); row={"block":block,"entry_date":e,"expiry_date":x,"stage_fit_history":{"B":"A","C":"A+B","VALIDATION":"A+B+C"}[block]}
        if not x or x not in block_dates: row.update({"bridge_calibration_eligible":False,"bridge_exclusion_reason":"block_boundary_or_missing_timing"}); bridge_rows.append(row); continue
        f=fore.loc[fore.model_day.dt.date.eq(e),"R_background_forecast"]; actual,reason=path_var(e,x,"development"); t0,t1=local_time(e,ASSUMED_IV_TIME),local_time(x,ASSUMED_EXPIRY_TIME); ann=horizon_events(t0,t1)
        row.update({"entry_time":t0,"expiry_time":t1,"horizon_class":"REGULAR" if (x-e).days<=REGULAR_HORIZON_MAX_CALENDAR_DAYS else "EXTENDED","R_background_forecast":f.iloc[0] if len(f)==1 else np.nan,"R_forecast_total_variance":f.iloc[0]/252 if len(f)==1 else np.nan,"actual_option_horizon_realised_variance":actual,**ann})
        if ann["event_timing_ambiguous"]: reason_code="ambiguous_event_timing"
        elif ann["exact_known_event_in_horizon"]: reason_code="known_event"
        else: reason_code=reason or ("forecast_unavailable" if not np.isfinite(row["R_forecast_total_variance"]) or row["R_forecast_total_variance"]<=0 else "")
        row["bridge_calibration_eligible"]=not bool(reason_code); row["bridge_exclusion_reason"]=reason_code; bridge_rows.append(row)
bridge=pd.DataFrame(bridge_rows)
assert not bridge.duplicated(["block","entry_date"]).any()
assert bridge.groupby("block").entry_date.apply(lambda x:not x.duplicated().any()).all()
assert (bridge.stage_fit_history==bridge.block.map({"B":"A","C":"A+B","VALIDATION":"A+B+C"})).all()
assert not bridge.block.eq("TEST").any()
bridge.to_csv(PROCESSED/"23_stage2_bridge_observations.csv",index=False)


In [6]:
def bridge_factor(blocks,stage):
    rows=[]
    for hc in ["REGULAR","EXTENDED"]:
        z=bridge.loc[bridge.block.isin(blocks)&bridge.horizon_class.eq(hc)&bridge.bridge_calibration_eligible]
        assert len(z)>=MIN_BRIDGE_CLASS_N,f"Bridge support insufficient: {stage}/{hc}={len(z)}"
        factor=z.actual_option_horizon_realised_variance.sum()/z.R_forecast_total_variance.sum(); assert np.isfinite(factor) and factor>0
        scope=bridge.loc[bridge.block.isin(blocks)&bridge.horizon_class.eq(hc)]
        rows.append({"application_stage":stage,"calibration_blocks":"+ ".join(blocks),"horizon_class":hc,"n_calibration":len(z),"sum_forecast_total_variance":z.R_forecast_total_variance.sum(),"sum_actual_option_variance":z.actual_option_horizon_realised_variance.sum(),"bridge_factor_b":factor,"median_actual_to_forecast_ratio":np.median(z.actual_option_horizon_realised_variance/z.R_forecast_total_variance),"known_event_horizons_excluded":int(scope.bridge_exclusion_reason.eq("known_event").sum()),"ambiguous_event_horizons_excluded":int(scope.bridge_exclusion_reason.eq("ambiguous_event_timing").sum()),"other_unavailable_horizons":int((~scope.bridge_calibration_eligible&~scope.bridge_exclusion_reason.isin(["known_event","ambiguous_event_timing"])).sum())})
    return rows
bridge_cal=pd.DataFrame(bridge_factor(["B"],"C")+bridge_factor(["B","C"],"VALIDATION")+bridge_factor(["B","C","VALIDATION"],"FUTURE_TEST"))
bridge_cal.to_csv(PROCESSED/"23_stage2_bridge_calibration.csv",index=False)
factors=bridge_cal.set_index(["application_stage","horizon_class"]).bridge_factor_b
info=[]
for r in raw.itertuples():
    d=r._asdict()
    d.pop("Index",None)
    if not d["economically_evaluable"]: info.append(d); continue
    stage="C" if d["development_role"]=="POLICY_TRAIN" else "VALIDATION"; factor=factors[(stage,d["horizon_class"])]; ivv=iv.loc[iv.date_key.eq(d["entry_date"]),"atm_iv"].iloc[0]; implied=(ivv/100)**2*d["T"]; forecast=factor*d["R_forecast_total_variance_unbridged"]
    assert ivv>0 and implied>0 and forecast>0
    d.update({"bridge_factor":factor,"forecast_option_horizon_variance":forecast,"raw_atm_iv":ivv,"implied_total_variance":implied,"forecast_vs_implied_log_ratio_G":np.log(forecast/implied)}); info.append(d)
info=pd.DataFrame(info); dev_info=info.loc[info.economically_evaluable].copy()
assert np.isfinite(dev_info.forecast_vs_implied_log_ratio_G).all()
assert np.allclose(dev_info.implied_total_variance,(dev_info.raw_atm_iv/100)**2*dev_info["T"])
dev_info.to_csv(PROCESSED/"23_stage2_development_information_set.csv",index=False)
annotations=info[["episode_id","known_event_in_horizon","n_known_events","known_event_families","known_event_timestamps","known_event_timestamp_statuses","event_timing_ambiguous"]]; annotations.to_csv(PROCESSED/"23_stage2_known_event_horizon_annotations.csv",index=False)
safe=frozen_test_safe.copy(); assert list(safe.columns)==SAFE_TEST_COLUMNS
assert not any(any(token in c.lower() for token in ["iv","sigma","forecast","bridge","variance","pnl","premium","payoff","return"]) for c in safe.columns)
safe.to_csv(PROCESSED/"23_stage2_test_safe_status.csv",index=False)
blocked=False
try: build_stage2_economic_row(stage1_test.iloc[0],"TEST","test")
except RuntimeError as e: assert str(e)==EMBARGO_MESSAGE; blocked=True
assert blocked
print("PASS — TEST Stage 2 economic-row builder blocked predictors before computation.")
support=[]
for label,x in [("C",C_candidates),("VALIDATION",V_candidates),("TEST_FROZEN_N21",pd.read_csv(PROCESSED/"21_hidden_event_daily_scores.csv").query("candidate_type != 'NONE'"))]:
    counts=x.candidate_type.value_counts(); ep=(C_eps if label=="C" else V_eps if label=="VALIDATION" else n21eps)
    support.append({"period":label,"REALISED_ONLY_candidates":counts.get("REALISED_ONLY",0),"IMPLIED_ONLY_candidates":counts.get("IMPLIED_ONLY",0),"total_candidates":len(x),"REALISED_ONLY_episodes":ep.candidate_type.eq("REALISED_ONLY").sum(),"IMPLIED_ONLY_episodes":ep.candidate_type.eq("IMPLIED_ONLY").sum(),"total_episodes":len(ep)})
support=pd.DataFrame(support); support.to_csv(PROCESSED/"23_stage2_hidden_event_support.csv",index=False)
meta23=pd.DataFrame([("notebook","23_option_application_development_information_set.ipynb"),("stage","Application 1 Stage 2"),("development_detector","EXPANDING_WINDOW_STAGED"),("abc_split","50/25/25 canonical chronology"),("model_specification_source","FROZEN_N19"),("hyperparameter_reselection","False"),("active_threshold",str(ACTIVE_THRESHOLD)),("min_model_support",str(MIN_MODEL_SUPPORT)),("canonical_implied_pairing","previous complete canonical model day"),("bridge_observation_unit","unique block x entry_date"),("event_timestamp_provenance","event_timestamp_original when available; otherwise DATE_ONLY_OR_TIMING_UNRESOLVED"),("bridge_formula","sum(actual option RV) / sum(R background forecast / 252)"),("bridge_classes","REGULAR | EXTENDED"),("bridge_known_event_exclusion","entire option horizon"),("bridge_forecasts","stage-specific OOS only"),("test_detector_recomputed","False"),("test_stage2_features_unlocked","False"),("trading_rule_defined","False"),("ml_policy_trained","False"),("event_weights_applied","False"),("strategy_pnl_computed","False")],columns=["field","value"]); meta23.to_csv(PROCESSED/"23_stage2_run_metadata.csv",index=False)
assert len(safe)==17 and int(safe.evaluable.sum())==16 and set(safe.episode_id)==set(stage1_test.episode_id)
assert TEST_STAGE2_FEATURES_UNLOCKED is False
assert (B_pairs.canonical_index-B_pairs.implied_reference_canonical_index).eq(1).all()
assert (C_pairs.canonical_index-C_pairs.implied_reference_canonical_index).eq(1).all()
assert (V_days.canonical_index-V_days.implied_reference_canonical_index).eq(1).all()
assert not bridge.duplicated(["block","entry_date"]).any()
econ_counts=pd.DataFrame([{"period":"C","economically_evaluable":int((raw.development_role.eq("POLICY_TRAIN")&raw.economically_evaluable).sum())},{"period":"VALIDATION","economically_evaluable":int((raw.development_role.eq("POLICY_VALIDATION")&raw.economically_evaluable).sum())},{"period":"TEST_FROZEN_N21","economically_evaluable":16}])
final_support=support.merge(econ_counts,on="period",how="left"); display(final_support)
display(bridge_cal[["application_stage","horizon_class","n_calibration","bridge_factor_b"]])
print(f"Detector calibration pairs: B={len(B_pairs)}, C={len(C_pairs)}; B q=({qloB:.6f}, {qhiB:.6f}); C q=({qloC:.6f}, {qhiC:.6f}).")
print("Stage 2 complete: TEST economic predictors LOCKED; trading policy NOT DEFINED; strategy PnL NOT COMPUTED.")


PASS — TEST Stage 2 economic-row builder blocked predictors before computation.


,period,REALISED_ONLY_candidates,IMPLIED_ONLY_candidates,total_candidates,REALISED_ONLY_episodes,IMPLIED_ONLY_episodes,total_episodes,economically_evaluable
0,C,9,0,9,9,0,9,8
1,VALIDATION,8,6,14,8,6,14,14
2,TEST_FROZEN_N21,4,14,18,4,13,17,16


,application_stage,horizon_class,n_calibration,bridge_factor_b
0,C,REGULAR,428,0.805086
1,C,EXTENDED,115,0.991743
2,VALIDATION,REGULAR,943,0.884261
3,VALIDATION,EXTENDED,264,0.881196
4,FUTURE_TEST,REGULAR,1643,0.820376
5,FUTURE_TEST,EXTENDED,484,0.838457


Detector calibration pairs: B=379, C=543; B q=(-2.775966, 2.538044); C q=(-2.502375, 2.555336).
Stage 2 complete: TEST economic predictors LOCKED; trading policy NOT DEFINED; strategy PnL NOT COMPUTED.
